# FaceRestore: train and score on Kaggle

Runs everything heavy on Kaggle's free GPU: download CelebA, prepare it, score
the existing models, fine-tune ours, and score it against them.
Optional: the app already ships CodeFormer, which beats upstream. Train only to
try to beat CodeFormer.

**Settings** (right panel): Accelerator **GPU T4 x1** (or P100), Internet **on**.
Then *Save Version* > *Save & Run All*. Outputs land in `/kaggle/working`:
`facerestore.pt`, `eval_before.json`, `eval_after.json`, `run/log.csv`.

Time: data ~15 min, scoring ~10 min each, training up to `MAX_MINUTES`.

In [ ]:
REPO = 'https://github.com/keshavdhingra001/FaceRestore'
BRANCH = 'main'        # set to a PR branch to try it before merging
STEPS = 40000          # ~2-3 h on a T4
MAX_MINUTES = 540      # stop and save before Kaggle's 12 h limit
EVAL_IMAGES = 2000

!git clone --depth 1 -b {BRANCH} {REPO} /kaggle/working/FaceRestore
%cd /kaggle/working/FaceRestore
!pip install -q lpips==0.1.4 pyarrow
import cv2, torch
print('opencv', cv2.__version__, '| torch', torch.__version__, '| gpu', torch.cuda.get_device_name(0))
assert hasattr(cv2, 'FaceDetectorYN'), 'OpenCV is too old for the YuNet face detector'

In [ ]:
!python scripts/fetch_weights.py --only detector upstream lama codeformer

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download('nielsr/CelebA-faces', repo_type='dataset', allow_patterns='data/*.parquet',
                  local_dir='/tmp/celeba')
!python scripts/prepare_celeba.py --parquet-dir /tmp/celeba/data --out /tmp/celeba256 \
    --detector weights/face_detection_yunet_2023mar.onnx --workers 4 | tail -4

## Score the existing models first
Held-out CelebA test faces, same holes for every model. LPIPS: lower is better.

In [ ]:
!python evaluate.py --data /tmp/celeba256 --weights weights --limit {EVAL_IMAGES} \
    --json /kaggle/working/eval_before.json

## Fine-tune ours from the upstream weights
Validates every 2000 steps; `best.pt` is the checkpoint with the lowest validation LPIPS.
To continue an earlier run, add its `last.pt` as a dataset and use `--resume`.

In [ ]:
!python train.py --data /tmp/celeba256 --init weights/upstream.pt --out /kaggle/working/run \
    --steps {STEPS} --batch-size 8 --workers 4 --max-minutes {MAX_MINUTES}

In [ ]:
!python evaluate.py --data /tmp/celeba256 --weights weights --engine ours=/kaggle/working/run/best.pt \
    --limit {EVAL_IMAGES} --json /kaggle/working/eval_after.json

In [ ]:
import shutil, hashlib
shutil.copy('/kaggle/working/run/best.pt', '/kaggle/working/facerestore.pt')
print('sha256', hashlib.sha256(open('/kaggle/working/facerestore.pt', 'rb').read()).hexdigest())

## Ship it
If `ours` beats `upstream` on LPIPS in `eval_after.json`:
1. Attach `facerestore.pt` to a GitHub release of the repo.
2. Put the printed sha256 into `scripts/fetch_weights.py` (`facerestore` entry).
3. Paste the score table into the README.